# Starflow AI Lab 002｜真实视频帧预测训练

**iPhone + 免费 Google Colab CPU + 不购买 API。**

上传你本人拍摄或获得授权的一段 5–10 秒 MP4 短视频，训练 10,579 参数的原创迷你卷积网络，从视频中相邻两帧预测下一帧。生成训练报告、预测 GIF 和模型权重。本实验不是通用 AI 视频生成、人物复刻，也不能生成原生 4K/120FPS 视频。

注意：上传的视频会进入 Google Colab 的远程会话，**不是仅留在 iPhone 本地**；避免上传隐私视频。免费运行环境可能限额或不可用。请勿购买计算单元。

## 第一步：上传你自己的 MP4 视频

In [ ]:
import cv2, numpy as np
from google.colab import files
uploaded = files.upload()
if not uploaded:
    raise ValueError("请先上传你有使用权的短视频")
filename = next(iter(uploaded))
cap = cv2.VideoCapture(filename)
if not cap.isOpened():
    raise ValueError("视频无法解码，请尝试 H.264 MP4")
fps = cap.get(cv2.CAP_PROP_FPS)
fps = fps if np.isfinite(fps) and fps > 1 else 25
step = max(1, round(fps / 5))
frames = []
count = 0
while count < min(int(fps * 10), 3000):
    ok, image = cap.read()
    if not ok:
        break
    if count % step == 0:
        image = cv2.resize(image, (48, 48), interpolation=cv2.INTER_AREA)
        frames.append(cv2.cvtColor(image, cv2.COLOR_BGR2RGB))
    count += 1
cap.release()
if len(frames) < 18:
    raise ValueError("视频可读帧数太少，请换一段 5–10 秒 MP4")
frames = np.asarray(frames, dtype=np.uint8)
print("读取成功：", len(frames), "个真实采样帧，48×48 像素")

## 第二步：使用 CPU 训练视频帧预测网络

In [ ]:
import torch, matplotlib.pyplot as plt
from torch import nn
from torch.utils.data import DataLoader, TensorDataset
torch.manual_seed(42)
torch.set_num_threads(2)
EPOCHS = 12
pixels = torch.from_numpy(frames.copy()).float().permute(0, 3, 1, 2) / 255.
x = torch.stack([torch.cat((pixels[i], pixels[i+1]), dim=0) for i in range(len(pixels)-2)])
y = pixels[2:]
split = max(12, min(len(x)-5, int(len(x)*.8)))
xtrain, ytrain, xtest, ytest = x[:split], y[:split], x[split:], y[split:]
class StarflowRealFrameV0(nn.Module):
    def __init__(self):
        super().__init__()
        self.net = nn.Sequential(
            nn.Conv2d(6, 16, 3, padding=1), nn.ReLU(),
            nn.Conv2d(16, 32, 3, padding=1), nn.ReLU(),
            nn.Conv2d(32, 16, 3, padding=1), nn.ReLU(),
            nn.Conv2d(16, 3, 3, padding=1))
    def forward(self, x):
        return x[:, 3:] + self.net(x)
model = StarflowRealFrameV0()
opt = torch.optim.Adam(model.parameters(), lr=0.001)
loader = DataLoader(TensorDataset(xtrain, ytrain), batch_size=8, shuffle=True)
baseline = ((xtest[:, 3:] - ytest)**2).mean().item()
best = float("inf")
best_weights = None
history = []
for epoch in range(EPOCHS):
    model.train()
    for xb, yb in loader:
        loss = ((model(xb) - yb)**2).mean()
        opt.zero_grad()
        loss.backward()
        opt.step()
    model.eval()
    with torch.no_grad():
        mse = ((model(xtest).clamp(0, 1) - ytest)**2).mean().item()
    history.append(mse)
    if mse < best:
        best = mse
        best_weights = {k:v.detach().clone() for k,v in model.state_dict().items()}
    print(f"训练轮次 {epoch+1:02d}/{EPOCHS} | 留出视频帧 MSE = {mse:.6f}")
model.load_state_dict(best_weights)
print("模型参数量：", sum(p.numel() for p in model.parameters()))
print("重复上一帧的基线 MSE：", baseline, "训练后最好 MSE：", best)
print("说明：来自同一个视频的测试片段不等于跨视频泛化验证")

## 第三步：评测、导出权重和预测 GIF（仅作研究验证）

In [ ]:
from PIL import Image
model.eval()
with torch.no_grad():
    predicted = model(xtest).clamp(0,1)
i = min(2, len(xtest)-1)
fig, axes = plt.subplots(1,3,figsize=(9,3))
images = [xtest[i,3:], predicted[i], ytest[i]]
for ax, img, title in zip(axes, images, ["previous", "Starflow prediction", "true next frame"]):
    ax.imshow(img.permute(1,2,0).numpy())
    ax.set_title(title)
    ax.axis("off")
plt.tight_layout()
plt.show()
plt.figure(figsize=(6,3))
plt.plot(history,label="held-out frame MSE")
plt.axhline(baseline,color="gray",linestyle="--",label="copy last frame baseline")
plt.legend()
plt.show()
gif = [Image.fromarray((predicted[j].permute(1,2,0).numpy()*255).astype("uint8")).resize((288,288)) for j in range(len(predicted))]
gif[0].save("starflow_real_frame_predictions.gif",save_all=True,append_images=gif[1:],duration=200,loop=0)
torch.save({"model_state_dict":model.state_dict(),"frame_size":48,"epochs":EPOCHS,"seed":42,"heldout_mse":best,"previous_frame_baseline_mse":baseline},"starflow_real_frame_v0.pt")
print("训练权重、预测 GIF 已保存")

## 可选：下载文件到 iPhone

In [ ]:
from google.colab import files
files.download("starflow_real_frame_v0.pt")
files.download("starflow_real_frame_predictions.gif")